In [1]:
# get home folder path
import os
import random
import ismrmrd
import h5py
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import xml.dom.minidom as minidom
import xml.dom.minidom
from collections import Counter
import torch

home_folder = os.path.expanduser("~")
parent_folder = os.path.abspath(os.path.join(home_folder, "..", ".."))
# cd to data folder
os.chdir(parent_folder)

data_folder = Path(parent_folder) / "data/datasets/msk_mri_h5/test"

h5_paths = sorted(
    p for p in data_folder.iterdir()
    if p.suffix.lower() in {'.h5', '.hdf5'} and p.is_file()
)
print(f'Found {len(h5_paths)} H5 files in: {data_folder}')

# choose random file from folder
random_file = random.choice(h5_paths)
print(f"Random file chosen from the data folder: {random_file}")

folder_size = sum(f.stat().st_size for f in data_folder.glob('**/*') if f.is_file()) / (1024 ** 3)
print(f"Size of the data folder: {folder_size:.2f} GB")

Found 429 H5 files in: /data/datasets/msk_mri_h5/test
Random file chosen from the data folder: /data/datasets/msk_mri_h5/test/meas_MID00330_FID129290_COR_T1_FS.h5
Size of the data folder: 387.63 GB


In [2]:
# --- config you may tweak ---
# Folder with full k-space .h5 files (you already set this up)
import os, shutil, h5py
from pathlib import Path
import numpy as np

home_folder = os.path.expanduser("~")
parent_folder = os.path.abspath(os.path.join(home_folder, "..", ".."))
os.chdir(parent_folder)

data_folder = Path(parent_folder) / "data/datasets/msk_mri_h5/train"  # k-space files
# Folder that holds the metadata-only .h5 (same basenames)
# EDIT THIS if your metadata-only outputs live somewhere else:
meta_folder = Path(home_folder) / "/home/paula/msk_mri_dataset/metadata_only/meta"

# Output folder (sibling to "test")
out_folder = Path(parent_folder) / "data/datasets/msk_mri_h5/train_new_metadata"
out_folder.mkdir(parents=True, exist_ok=True)
print("kspace src :", data_folder)
print("meta src   :", meta_folder)
print("dest out   :", out_folder)

# groups/datasets that store k-space and must be preserved in the k-space copy
DATASET_DATA_KEYS = {"data", "rawdata", "data0"}

def copy_attrs(src_obj, dst_obj):
    dst_obj.attrs.clear()
    for k, v in src_obj.attrs.items():
        dst_obj.attrs[k] = v

def read_xml_bytes(meta_h5: Path) -> bytes | None:
    with h5py.File(meta_h5, "r") as f:
        ds = f.get("dataset")
        if ds is None or "xml" not in ds:
            return None
        x = ds["xml"]
        # accept scalar or len-1 vlen
        if x.shape == ():
            val = x[()]
            return val.encode("utf-8") if isinstance(val, str) else bytes(val)
        # len-1 vlen bytes
        v = x[0]
        return v if isinstance(v, (bytes, bytearray)) else bytes(v)

def write_xml_len1_bytes(h5_path: Path, xml_bytes: bytes):
    with h5py.File(h5_path, "r+") as f:
        ds = f.require_group("dataset")
        if "xml" in ds:
            del ds["xml"]
        vlen_bytes = h5py.special_dtype(vlen=bytes)
        arr = np.empty((1,), dtype=object)
        arr[0] = xml_bytes
        ds.create_dataset("xml", data=arr, dtype=vlen_bytes)  # shape (1,), vlen bytes

def replace_top_level_metadata(dst_path: Path, meta_path: Path):
    """
    Replace all top-level metadata groups/datasets in dst with those from meta,
    except 'dataset' (handled separately).
    """
    with h5py.File(meta_path, "r") as fmeta, h5py.File(dst_path, "r+") as fdst:
        # copy file-level attrs
        copy_attrs(fmeta, fdst)

        for key in fmeta.keys():
            if key == "dataset":
                continue  # handled separately
            # remove old key in dst (if exists), then copy fresh from meta
            if key in fdst:
                del fdst[key]
            fmeta.copy(key, fdst, name=key)

def replace_dataset_metadata(dst_path: Path, meta_path: Path):
    """
    Within /dataset, copy everything from meta except k-space datasets,
    and overwrite xml using the same (1,) vlen bytes layout.
    """
    xml_bytes = read_xml_bytes(meta_path)
    with h5py.File(meta_path, "r") as fmeta, h5py.File(dst_path, "r+") as fdst:
        md = fmeta.get("dataset")
        if md is None:
            # no dataset group in meta; nothing to do
            return
        dd = fdst.require_group("dataset")

        # Overwrite /dataset/xml first (as len-1 vlen bytes)
        if xml_bytes:
            if "xml" in dd:
                del dd["xml"]
            vlen_bytes = h5py.special_dtype(vlen=bytes)
            arr = np.empty((1,), dtype=object)
            arr[0] = xml_bytes
            dd.create_dataset("xml", data=arr, dtype=vlen_bytes)

        # For each item under /dataset in meta: copy unless it's a k-space dataset
        for name, obj in md.items():
            if name in DATASET_DATA_KEYS:
                # keep the original data/rawdata/data0 from the k-space file
                continue
            if name == "xml":
                continue  # already handled
            # remove existing in dst if present
            if name in dd:
                del dd[name]
            # copy group or dataset
            md.copy(name, dd, name=name)

        # copy attrs on /dataset group itself from meta
        copy_attrs(md, dd)

def human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024 or u == "TB": return f"{n:.1f}{u}"
        n /= 1024

kspace_paths = sorted(p for p in data_folder.iterdir() if p.suffix.lower() in {".h5", ".hdf5"} and p.is_file())
print(f"Found {len(kspace_paths)} k-space files.")

ok = miss = fail = 0
for kp in kspace_paths:
    meta = meta_folder / kp.name
    if not meta.exists():
        print(f"[SKIP] missing meta for {kp.name}")
        miss += 1
        continue

    dst = out_folder / kp.name
    try:
        shutil.copy2(kp, dst)  # copy full file first
        # replace top-level metadata sections
        replace_top_level_metadata(dst, meta)
        # replace /dataset/* metadata (not the k-space)
        replace_dataset_metadata(dst, meta)

        # quick sanity for your legacy access pattern:
        with h5py.File(dst, "r") as f:
            xml_ds = f["dataset"]["xml"]
            assert xml_ds.shape == (1,), f"xml shape {xml_ds.shape} != (1,)"
            assert isinstance(xml_ds[0], (bytes, bytearray)), "xml[0] must be bytes"

        print(f"[OK] {kp.name} → {dst.parent.name}  ({human(dst.stat().st_size)})")
        ok += 1
    except Exception as e:
        print(f"[ERROR] {kp.name}: {e}")
        fail += 1

print(f"\nSummary: ok={ok}  missing_meta={miss}  failed={fail}")
print(f"Output folder: {out_folder}")

kspace src : /data/datasets/msk_mri_h5/train
meta src   : /home/paula/msk_mri_dataset/metadata_only/meta
dest out   : /data/datasets/msk_mri_h5/train_new_metadata
Found 1823 k-space files.
[OK] meas_MID00015_FID117913_COR_T2.h5 → train_new_metadata  (1.1GB)
[OK] meas_MID00016_FID118817_COR_T2.h5 → train_new_metadata  (946.0MB)
[OK] meas_MID00016_FID120712_COR_T2.h5 → train_new_metadata  (397.6MB)
[OK] meas_MID00017_FID116488_RT_3_PLANE_UOF.h5 → train_new_metadata  (161.8MB)
[OK] meas_MID00017_FID118818_SAG_T2.h5 → train_new_metadata  (946.0MB)
[OK] meas_MID00017_FID126574_RT_3_PLANE_UOF.h5 → train_new_metadata  (161.8MB)
[OK] meas_MID00018_FID110986_COR_PD_FS.h5 → train_new_metadata  (949.7MB)
[OK] meas_MID00018_FID111995_COR_PD_FS.h5 → train_new_metadata  (910.1MB)
[OK] meas_MID00018_FID117916_SAG_T2.h5 → train_new_metadata  (1.2GB)
[OK] meas_MID00018_FID118819_AXIAL_T2.h5 → train_new_metadata  (808.9MB)
[OK] meas_MID00019_FID117917_SAG_T1.h5 → train_new_metadata  (825.5MB)
[OK] meas_